<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px; float: right;">
    </div>
</a>

# **Factory Optimization with cuOpt**

<h2><b>Use Case 1:</b> Autoclave Scheduling with MIP</h2>

In this exercise, you will model a simplified autoclave scheduling problem using cuOpt's algebraic modeling API.

This first version uses a **baseline hard-deadline formulation**:

> Cure as many feasible parts as possible before their out-time deadline. Parts not cured by that deadline are treated as missed/scrap risk.


<hr>

# **Mixed-Integer Programming for Autoclave Scheduling**

Mixed-Integer Programming (MIP) models express an optimization problem using decision variables, a linear objective, and linear constraints.

What makes this model a MIP is the presence of **binary** yes/no decisions.

For this factory use case, the yes/no decision is:

> Should this part go into this autoclave run?

The model will decide which parts to cure while respecting part readiness, out-time deadlines, autoclave capacity, and cure compatibility.

In cuOpt, we will build this model with:

1. `Problem(...)` to initialize the optimization model.
2. `addVariable(...)` to define decision variables.
3. `addConstraint(...)` to add factory rules.
4. `setObjective(...)` to define what "best" means.
5. `solve()` to optimize the model.

## Problem Definition

A factory has composite parts waiting to be cured in an autoclave.

Think of the autoclave as a large oven/pressure vessel. It can cure several compatible parts at the same time, but it has limited space.

Each part has:

1. `ready_hr`: the earliest time it can start cure.
2. `deadline_hr`: the latest safe finish time before its out-time expires.
3. `cure_time_hr`: how long the cure takes.
4. `priority`: a tie-breaker when not everything can fit.
5. `material` and cure time, which we turn into a simple cure recipe.

In this first model, we use **one autoclave**. Later, we will add a second autoclave and compare what changes.

## Baseline Hard-Deadline Formulation

This formulation answers this question:

> Which parts can we cure before their out-time deadline, given limited autoclave capacity?

This version allows some parts to miss the deadline. Those parts are not simply waiting for tomorrow; in this simplified scenario, they are the parts **not cured before the out-time deadline**.

### Decision Variables

For each part and each possible autoclave run:

> `x[part, recipe, start] = 1` if that part is assigned to that run.

> `x[part, recipe, start] = 0` otherwise.

We also use a helper variable:

> `y[recipe, start] = 1` if the autoclave runs that recipe at that start time.

### Objective

Primary goal:

> Maximize the number of parts cured before the out-time deadline.

Tie-breaker:

> Prefer higher-priority parts when two schedules cure the same number of parts.

### Constraints

1. Each part can be cured at most once.
2. Each autoclave run has limited capacity.
3. A part cannot be assigned before it is ready.
4. A part must finish before its out-time deadline.
5. Only compatible parts can cure together.
6. One autoclave can run only one recipe at a time.

For this teaching model, we create candidate autoclave runs every few hours. This keeps the MIP easy to read.

## The MIP Model In Math

Let:

- `P` be the set of parts.
- `R` be the set of cure recipes.
- `T` be the set of candidate start times.
- `C` be the max number of parts in one autoclave run.
- `w_p = 1000 + priority_p` be the score for curing part `p`.

Decision variables:

> $x_{p,r,t} = 1$ if part $p$ is assigned to recipe $r$ at start time $t$.

> $y_{r,t} = 1$ if the autoclave runs recipe $r$ at start time $t$.

Objective:

> maximize cured parts, then use priority as a tie-breaker

$$
\max \sum_{p \in P} \sum_{r \in R} \sum_{t \in T} w_p x_{p,r,t}
$$

Core constraints:

Each part can be cured at most once:

$$
\sum_{r \in R} \sum_{t \in T} x_{p,r,t} \le 1 \qquad p \in P
$$

Each run has limited capacity:

$$
\sum_{p \in P} x_{p,r,t} \le C y_{r,t} \qquad r \in R,\ t \in T
$$

Only one recipe can run at a start time:

$$
\sum_{r \in R} y_{r,t} \le 1 \qquad t \in T
$$

The ready-time and out-time deadline rules are handled when we create variables: if a part cannot physically fit in that run, we do not create that `x` variable.

Let's start by loading the data for the autoclave use case.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("data")

parts = pd.read_csv(DATA_DIR / "autoclave_parts.csv")
autoclaves = pd.read_csv(DATA_DIR / "autoclaves.csv")

print(f"Loaded {len(parts)} parts")
print(f"Loaded {len(autoclaves)} autoclaves")

The parts table is the list of jobs we may cure.

Key columns:

- `ready_hr`: earliest time the part can enter cure.
- `deadline_hr`: latest safe finish time before out-time expires.
- `cure_time_hr`: how long the cure run takes.
- `priority`: tie-breaker in the objective.
- `area_m2`: footprint data for a later capacity-by-area extension.

In [ ]:
display(parts)

The autoclaves table describes each autoclave's capacity and when it is available.

For the base model, we use `batch_capacity`, which means max number of parts in one run.

The `usable_area_m2` column is included for a later extension where capacity is based on footprint instead of part count.

In [ ]:
display(autoclaves)

## Prepare The First Scenario

We will start with only one autoclave: `AC-1`.

The current data has material and cure time. For the first workshop model, we combine those into a simple `recipe_id`.

Parts with the same `recipe_id` are allowed to cure together.

This is a simplification, but it teaches the right modeling move: compatibility becomes a rule in the optimization model.

In [ ]:
scenario_parts = parts.copy()
scenario_parts["recipe_id"] = (
    scenario_parts["material"]
    + "_cure_"
    + scenario_parts["cure_time_hr"].map(lambda value: f"{value:g}h")
)

active_autoclave = autoclaves.query("autoclave_id == 'AC-1'").iloc[0]

print("Active autoclave:", active_autoclave.autoclave_id)
print("Run capacity:", int(active_autoclave.batch_capacity), "parts")
print("Candidate run spacing:", int(active_autoclave.slot_duration_hr), "hours")

display(scenario_parts[[
    "part_id", "part_type", "ready_hr", "deadline_hr",
    "cure_time_hr", "priority", "recipe_id"
]])

<b>Let us visualize the part timing windows.</b>

Each bar shows when a part is available to be cured. The red marker is the out-time deadline.

In [ ]:
import matplotlib.pyplot as plt


def draw_part_windows(part_df):
    plot_df = part_df.sort_values(["deadline_hr", "ready_hr", "part_id"]).reset_index(drop=True)

    fig_height = max(4, 0.35 * len(plot_df))
    fig, ax = plt.subplots(figsize=(10, fig_height))

    for row_index, row in plot_df.iterrows():
        window_width = float(row.deadline_hr) - float(row.ready_hr)
        ax.barh(
            row_index,
            window_width,
            left=float(row.ready_hr),
            height=0.55,
            color="#76B900",
            alpha=0.55,
            edgecolor="black",
        )
        ax.scatter(float(row.deadline_hr), row_index, marker="|", s=180, color="#d62728")
        ax.text(
            float(row.ready_hr) + 0.05,
            row_index,
            f"{row.part_id} ({row.recipe_id.split('_')[-1]})",
            va="center",
            fontsize=8,
        )

    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df["part_id"])
    ax.set_xlabel("Hours from now")
    ax.set_title("Ready Time To Out-Time Deadline By Part")
    ax.grid(axis="x", alpha=0.25)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


draw_part_windows(scenario_parts)

<hr>

## **Step 1:** Initialize The Optimization Problem

Let us name our model **"Autoclave Baseline Hard Deadline MIP"** and create a `Problem` object.

In [ ]:
from cuopt.linear_programming.problem import Problem, INTEGER, MAXIMIZE

problem = Problem("Autoclave Baseline Hard Deadline MIP")

<hr>

## **Step 2:** Define Candidate Autoclave Runs

A run is one time the autoclave turns on.

For this draft, we create possible runs every `slot_duration_hr` hours. Since the current cure times are shorter than this spacing, the runs do not overlap.

In [ ]:
start_hr = int(active_autoclave.available_from_hr)
end_hr = 24
step_hr = int(active_autoclave.slot_duration_hr)

run_starts = list(range(start_hr, end_hr + 1, step_hr))
recipes = sorted(scenario_parts["recipe_id"].unique())

candidate_runs = pd.DataFrame(
    [
        {"recipe_id": recipe_id, "start_hr": run_start}
        for recipe_id in recipes
        for run_start in run_starts
    ]
)

print(f"Candidate start times: {run_starts}")
print(f"Recipes: {recipes}")
display(candidate_runs.head(12))

<hr>

## **Step 3:** Add Variables

We add two sets of binary variables.

1. `y[recipe, start]`: whether the autoclave runs that recipe at that start time.
2. `x[part, recipe, start]`: whether a part is assigned to that run.

To keep the model smaller, we only create `x` variables for choices that are physically possible:

- the part is ready by the start time
- the part can finish by its deadline
- the part recipe matches the run recipe


In [ ]:
def safe_name(value):
    return str(value).replace("-", "_").replace(".", "p").replace(" ", "_")

run_used = {}
for _, run in candidate_runs.iterrows():
    key = (run.recipe_id, int(run.start_hr))
    run_used[key] = problem.addVariable(
        lb=0,
        ub=1,
        vtype=INTEGER,
        name=f"y_{safe_name(run.recipe_id)}_{int(run.start_hr)}",
    )

assign = {}
for _, part in scenario_parts.iterrows():
    for run_start in run_starts:
        finish_hr = run_start + float(part.cure_time_hr)

        if run_start < float(part.ready_hr):
            continue
        if finish_hr > float(part.deadline_hr):
            continue

        key = (part.part_id, part.recipe_id, int(run_start))
        assign[key] = problem.addVariable(
            lb=0,
            ub=1,
            vtype=INTEGER,
            name=f"x_{safe_name(part.part_id)}_{safe_name(part.recipe_id)}_{int(run_start)}",
        )

print(f"Added {len(run_used)} run variables")
print(f"Added {len(assign)} feasible assignment variables")

The assignment variables are the main decisions in this model.

Here are a few of the feasible choices cuOpt is allowed to consider.

In [ ]:
assignment_preview = pd.DataFrame(
    [
        {"part_id": part_id, "recipe_id": recipe_id, "start_hr": run_start}
        for part_id, recipe_id, run_start in assign.keys()
    ]
).sort_values(["part_id", "start_hr"]).head(10)

display(assignment_preview)

Notice what happened here: impossible choices were never added to the model.

For example, if a part is not ready yet, or if it would miss its out-time deadline, that assignment variable does not exist.

This is a common modeling technique. Instead of creating every possible variable and then forbidding many of them with constraints, we create only the decisions that could physically happen.

#### **Understanding Component Arithmetic**

The original MIP workbook shows that cuOpt variables can be combined with normal Python arithmetic to build linear expressions.

That is what lets us write readable constraints such as:

> `sum(choices) <= 1`

instead of manually building matrix rows.

In [ ]:
sample_part_id = scenario_parts.iloc[0]["part_id"]
sample_choices = [
    variable
    for (candidate_part_id, recipe_id, run_start), variable in assign.items()
    if candidate_part_id == sample_part_id
]

print(f"Sample part: {sample_part_id}")
print(f"Feasible assignment variables for this part: {len(sample_choices)}")
print("Example variable name:", sample_choices[0].getVariableName())
print("Constraint shape: sum(all feasible assignment variables for this part) <= 1")

<hr>

## **Step 4:** Add Constraints

Next, we add the factory rules.

In cuOpt, constraints can be written using algebraic expressions such as:

> `sum(choices) <= 1`

or

> `sum(choices) <= capacity * y`

cuOpt turns these expressions into the matrix form that the MIP solver optimizes.

### Constraint 1: Each Part Can Be Cured At Most Once

Because this is the baseline hard-deadline model, a part is allowed to be left out.

That is why we use `<= 1` instead of `== 1`.

In [ ]:
part_assignment_constraints = []

for part_id in scenario_parts["part_id"]:
    choices = [
        variable
        for (candidate_part_id, recipe_id, run_start), variable in assign.items()
        if candidate_part_id == part_id
    ]

    if choices:
        constraint = problem.addConstraint(
            sum(choices) <= 1,
            name=f"at_most_once_{safe_name(part_id)}",
        )
        part_assignment_constraints.append(constraint)

print(f"Added {len(part_assignment_constraints)} part assignment constraints")

### Constraint 2: Each Run Has Limited Capacity

The autoclave can only hold a fixed number of parts in one run.

In [ ]:
capacity_constraints = []
activation_constraints = []
run_capacity = int(active_autoclave.batch_capacity)

for recipe_id in recipes:
    for run_start in run_starts:
        choices = [
            variable
            for (part_id, candidate_recipe_id, candidate_start), variable in assign.items()
            if candidate_recipe_id == recipe_id and candidate_start == run_start
        ]

        y = run_used[(recipe_id, run_start)]

        capacity_constraints.append(
            problem.addConstraint(
                sum(choices) <= run_capacity * y,
                name=f"capacity_{safe_name(recipe_id)}_{run_start}",
            )
        )

        if choices:
            activation_constraints.append(
                problem.addConstraint(
                    y <= sum(choices),
                    name=f"activate_only_if_used_{safe_name(recipe_id)}_{run_start}",
                )
            )
        else:
            activation_constraints.append(
                problem.addConstraint(
                    y == 0,
                    name=f"no_feasible_parts_{safe_name(recipe_id)}_{run_start}",
                )
            )

print(f"Added {len(capacity_constraints)} capacity constraints")
print(f"Added {len(activation_constraints)} run activation constraints")

### Constraint 3: One Autoclave Run Uses One Recipe

At a given start time, `AC-1` can run only one compatible recipe.

In [ ]:
recipe_choice_constraints = []

for run_start in run_starts:
    recipe_choice_constraints.append(
        problem.addConstraint(
            sum(run_used[(recipe_id, run_start)] for recipe_id in recipes) <= 1,
            name=f"one_recipe_at_start_{run_start}",
        )
    )

print(f"Added {len(recipe_choice_constraints)} recipe choice constraints")

At this point, the model knows the most important rules:

- a part cannot be cured more than once
- a run cannot exceed autoclave capacity
- incompatible recipes cannot be mixed in the same run
- ready time and out-time deadline are handled by only creating feasible assignment variables

In MIP language, variables are often called **columns**, and constraints are often called **rows**. Larger real-world models may have thousands or millions of each.

### Model Size Check

This is a tiny training problem, but cuOpt still sees it as a mathematical program with variables and constraints.

In [ ]:
one_ac_model_size = pd.DataFrame([
    {"model_piece": "run variables", "count": len(run_used)},
    {"model_piece": "assignment variables", "count": len(assign)},
    {"model_piece": "part assignment constraints", "count": len(part_assignment_constraints)},
    {"model_piece": "capacity constraints", "count": len(capacity_constraints)},
    {"model_piece": "run activation constraints", "count": len(activation_constraints)},
    {"model_piece": "recipe choice constraints", "count": len(recipe_choice_constraints)},
])

display(one_ac_model_size)

<hr>

## **Step 5:** Set The Objective

The objective represents the goal of the optimization.

For this baseline formulation, the primary goal is to cure as many parts as possible before their out-time deadlines.

We also add a small priority tie-breaker so that when two schedules cure the same number of parts, the model prefers higher-priority parts.

The `1000 + priority` coefficient is intentional. It makes one additional cured part more valuable than any priority tie-breaker.

In [ ]:
priority_by_part = scenario_parts.set_index("part_id")["priority"].to_dict()

# 1000 is intentionally larger than the total possible priority score.
# This makes "cure one more part" more important than any priority tie-breaker.
objective = sum(
    (1000 + priority_by_part[part_id]) * variable
    for (part_id, recipe_id, run_start), variable in assign.items()
)

problem.setObjective(objective, sense=MAXIMIZE)

<hr>

## **Step 6:** Optimize

We have successfully modeled the first autoclave scheduling problem in cuOpt.

When you run the next cell, cuOpt will print solver information such as rows, columns, nonzeros, presolve progress, objective value, and solve time.

For this small workshop model, the solve should be fast. On a real factory-sized MIP, the same modeling pattern can become much larger.

In [ ]:
problem.solve()

<hr>

## **Step 7:** Analyze The Results

Now we inspect which parts were assigned to autoclave runs and which parts were not cured before their out-time deadline.

In [ ]:
solution_rows = []

for (part_id, recipe_id, run_start), variable in assign.items():
    if variable.Value > 0.5:
        part = scenario_parts.query("part_id == @part_id").iloc[0]
        solution_rows.append(
            {
                "part_id": part_id,
                "part_type": part.part_type,
                "recipe_id": recipe_id,
                "start_hr": run_start,
                "finish_hr": run_start + float(part.cure_time_hr),
                "deadline_hr": float(part.deadline_hr),
                "priority": int(part.priority),
            }
        )

solution = pd.DataFrame(solution_rows).sort_values(
    ["start_hr", "recipe_id", "priority"],
    ascending=[True, True, False],
)

cured_parts = set(solution["part_id"])
missed_out_time_deadline = scenario_parts[
    ~scenario_parts["part_id"].isin(cured_parts)
].copy()

print("|============================================================|")
print("|                    Solution Metadata                       |")
print("|============================================================|")
print(f"Objective value: {problem.ObjValue:.2f}")
print(f"Solve time: {problem.SolveTime:.3f} seconds")
print(f"Cured parts: {len(solution)} of {len(scenario_parts)}")
print(f"Parts not cured before out-time deadline: {len(missed_out_time_deadline)}")

print("\nOptimized schedule:")
display(solution)

print("Parts not cured before out-time deadline:")
display(missed_out_time_deadline[[
    "part_id", "part_type", "ready_hr", "deadline_hr",
    "cure_time_hr", "priority", "recipe_id"
]])

### Visualize The One-Autoclave Schedule

The table is useful, but the schedule is easier to understand as a timeline.

In [ ]:
def plot_autoclave_schedule(schedule_df, title):
    if schedule_df.empty:
        print("No cured parts to plot.")
        return

    plot_df = schedule_df.copy()
    if "autoclave_id" not in plot_df.columns:
        plot_df["autoclave_id"] = "AC-1"

    plot_df = plot_df.sort_values(
        ["start_hr", "autoclave_id", "recipe_id", "part_id"]
    ).reset_index(drop=True)

    recipe_ids = sorted(plot_df["recipe_id"].unique())
    palette = plt.cm.Set2.colors
    recipe_colors = {
        recipe_id: palette[index % len(palette)]
        for index, recipe_id in enumerate(recipe_ids)
    }

    fig_height = max(4, 0.35 * len(plot_df))
    fig, ax = plt.subplots(figsize=(11, fig_height))

    for row_index, row in plot_df.iterrows():
        duration = float(row.finish_hr) - float(row.start_hr)
        ax.barh(
            row_index,
            duration,
            left=float(row.start_hr),
            height=0.55,
            color=recipe_colors[row.recipe_id],
            edgecolor="black",
        )
        ax.scatter(float(row.deadline_hr), row_index, marker="|", s=180, color="#d62728")
        ax.text(
            float(row.start_hr) + 0.05,
            row_index,
            f"{row.part_id} / {row.autoclave_id}",
            va="center",
            fontsize=8,
        )

    ax.set_yticks(range(len(plot_df)))
    ax.set_yticklabels(plot_df["part_id"])
    ax.set_xlabel("Hours from now")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


plot_autoclave_schedule(solution.assign(autoclave_id="AC-1"), "One-Autoclave Cure Schedule")

The missed-deadline list is important.

In this baseline formulation, a part listed here does not mean the model failed. It means the model found that, under the current capacity and timing rules, that part could not be included in the best on-time cure schedule.

Because this is an out-time deadline, the operational meaning is serious: these are parts that would need escalation, rework, or scrap handling in the real process.

<hr>

## **Extra Section:** Add A Second Autoclave

The first model used only `AC-1`.

Now we will use both autoclaves in `data/autoclaves.csv`.

The modeling idea is the same, but the decision variable gets one more dimension:

> `x[part, autoclave, recipe, start] = 1` if that part is cured in that autoclave run.

This lets the factory run two compatible cure recipes at the same time, as long as they are on different autoclaves.

### Compare The Available Autoclaves

Before changing the model, look at what capacity we are adding.

In [ ]:
active_autoclaves = autoclaves.copy()

display(active_autoclaves[[
    "autoclave_id", "batch_capacity", "usable_area_m2",
    "available_from_hr", "slot_duration_hr"
]])

### Build Candidate Runs For Each Autoclave

The one-autoclave model had candidate runs by recipe and start time.

The two-autoclave model has candidate runs by autoclave, recipe, and start time.

In [ ]:
candidate_runs_two_ac = []

for _, autoclave in active_autoclaves.iterrows():
    autoclave_run_starts = list(range(
        int(autoclave.available_from_hr),
        end_hr + 1,
        int(autoclave.slot_duration_hr),
    ))

    for recipe_id in recipes:
        for run_start in autoclave_run_starts:
            candidate_runs_two_ac.append({
                "autoclave_id": autoclave.autoclave_id,
                "recipe_id": recipe_id,
                "start_hr": run_start,
                "batch_capacity": int(autoclave.batch_capacity),
            })

candidate_runs_two_ac = pd.DataFrame(candidate_runs_two_ac)

print(f"Candidate runs with one autoclave: {len(candidate_runs)}")
print(f"Candidate runs with two autoclaves: {len(candidate_runs_two_ac)}")
display(candidate_runs_two_ac.head(12))

### Rebuild The MIP With The Autoclave Dimension

Instead of editing the first model in place, we build a second model.

This makes the comparison easier to read.

In [ ]:
two_autoclave_problem = Problem("Autoclave Baseline Hard Deadline MIP - Two Autoclaves")

two_ac_run_used = {}
for _, run in candidate_runs_two_ac.iterrows():
    key = (run.autoclave_id, run.recipe_id, int(run.start_hr))
    two_ac_run_used[key] = two_autoclave_problem.addVariable(
        lb=0,
        ub=1,
        vtype=INTEGER,
        name=(
            f"y_{safe_name(run.autoclave_id)}_"
            f"{safe_name(run.recipe_id)}_{int(run.start_hr)}"
        ),
    )

two_ac_assign = {}
for _, part in scenario_parts.iterrows():
    for _, run in candidate_runs_two_ac.iterrows():
        run_start = int(run.start_hr)
        finish_hr = run_start + float(part.cure_time_hr)

        if part.recipe_id != run.recipe_id:
            continue
        if run_start < float(part.ready_hr):
            continue
        if finish_hr > float(part.deadline_hr):
            continue

        key = (part.part_id, run.autoclave_id, part.recipe_id, run_start)
        two_ac_assign[key] = two_autoclave_problem.addVariable(
            lb=0,
            ub=1,
            vtype=INTEGER,
            name=(
                f"x_{safe_name(part.part_id)}_{safe_name(run.autoclave_id)}_"
                f"{safe_name(part.recipe_id)}_{run_start}"
            ),
        )

print(f"Added {len(two_ac_run_used)} two-autoclave run variables")
print(f"Added {len(two_ac_assign)} feasible two-autoclave assignment variables")

### Add The Same Core Constraints

The constraints are the same ideas as before:

1. each part can be cured at most once
2. each autoclave run has limited capacity
3. each autoclave can run only one recipe at a time

In [ ]:
two_ac_part_constraints = []

for part_id in scenario_parts["part_id"]:
    choices = [
        variable
        for (candidate_part_id, autoclave_id, recipe_id, run_start), variable
        in two_ac_assign.items()
        if candidate_part_id == part_id
    ]

    if choices:
        two_ac_part_constraints.append(
            two_autoclave_problem.addConstraint(
                sum(choices) <= 1,
                name=f"two_ac_at_most_once_{safe_name(part_id)}",
            )
        )

two_ac_capacity_constraints = []
two_ac_activation_constraints = []
capacity_by_autoclave = active_autoclaves.set_index("autoclave_id")["batch_capacity"].to_dict()

for _, run in candidate_runs_two_ac.iterrows():
    autoclave_id = run.autoclave_id
    recipe_id = run.recipe_id
    run_start = int(run.start_hr)
    capacity = int(capacity_by_autoclave[autoclave_id])
    y = two_ac_run_used[(autoclave_id, recipe_id, run_start)]

    choices = [
        variable
        for (part_id, candidate_autoclave_id, candidate_recipe_id, candidate_start), variable
        in two_ac_assign.items()
        if (
            candidate_autoclave_id == autoclave_id
            and candidate_recipe_id == recipe_id
            and candidate_start == run_start
        )
    ]

    two_ac_capacity_constraints.append(
        two_autoclave_problem.addConstraint(
            sum(choices) <= capacity * y,
            name=f"two_ac_capacity_{safe_name(autoclave_id)}_{safe_name(recipe_id)}_{run_start}",
        )
    )

    if choices:
        two_ac_activation_constraints.append(
            two_autoclave_problem.addConstraint(
                y <= sum(choices),
                name=f"two_ac_activate_only_if_used_{safe_name(autoclave_id)}_{safe_name(recipe_id)}_{run_start}",
            )
        )
    else:
        two_ac_activation_constraints.append(
            two_autoclave_problem.addConstraint(
                y == 0,
                name=f"two_ac_no_feasible_parts_{safe_name(autoclave_id)}_{safe_name(recipe_id)}_{run_start}",
            )
        )

two_ac_recipe_choice_constraints = []
for autoclave_id in active_autoclaves["autoclave_id"]:
    starts_for_autoclave = sorted(
        candidate_runs_two_ac.query("autoclave_id == @autoclave_id")["start_hr"].unique()
    )

    for run_start in starts_for_autoclave:
        two_ac_recipe_choice_constraints.append(
            two_autoclave_problem.addConstraint(
                sum(
                    two_ac_run_used[(autoclave_id, recipe_id, int(run_start))]
                    for recipe_id in recipes
                ) <= 1,
                name=f"two_ac_one_recipe_{safe_name(autoclave_id)}_{int(run_start)}",
            )
        )

print(f"Added {len(two_ac_part_constraints)} part assignment constraints")
print(f"Added {len(two_ac_capacity_constraints)} capacity constraints")
print(f"Added {len(two_ac_recipe_choice_constraints)} recipe choice constraints")

### Solve The Two-Autoclave Model

We use the same objective as the base model: cure as many parts as possible, with priority as a tie-breaker.

In [ ]:
two_ac_objective = sum(
    (1000 + priority_by_part[part_id]) * variable
    for (part_id, autoclave_id, recipe_id, run_start), variable
    in two_ac_assign.items()
)

two_autoclave_problem.setObjective(two_ac_objective, sense=MAXIMIZE)
two_autoclave_problem.solve()

### Analyze The Two-Autoclave Schedule

In [ ]:
two_ac_solution_rows = []

for (part_id, autoclave_id, recipe_id, run_start), variable in two_ac_assign.items():
    if variable.Value > 0.5:
        part = scenario_parts.query("part_id == @part_id").iloc[0]
        two_ac_solution_rows.append({
            "part_id": part_id,
            "part_type": part.part_type,
            "autoclave_id": autoclave_id,
            "recipe_id": recipe_id,
            "start_hr": run_start,
            "finish_hr": run_start + float(part.cure_time_hr),
            "deadline_hr": float(part.deadline_hr),
            "priority": int(part.priority),
        })

two_ac_solution = pd.DataFrame(two_ac_solution_rows).sort_values(
    ["start_hr", "autoclave_id", "recipe_id", "priority"],
    ascending=[True, True, True, False],
)

two_ac_cured_parts = set(two_ac_solution["part_id"])
two_ac_missed_out_time_deadline = scenario_parts[
    ~scenario_parts["part_id"].isin(two_ac_cured_parts)
].copy()

print("|============================================================|")
print("|              Two-Autoclave Solution Metadata               |")
print("|============================================================|")
print(f"Objective value: {two_autoclave_problem.ObjValue:.2f}")
print(f"Solve time: {two_autoclave_problem.SolveTime:.3f} seconds")
print(f"Cured parts: {len(two_ac_solution)} of {len(scenario_parts)}")
print(f"Parts not cured before out-time deadline: {len(two_ac_missed_out_time_deadline)}")

print("\nOptimized two-autoclave schedule:")
display(two_ac_solution)

print("Parts not cured before out-time deadline with two autoclaves:")
display(two_ac_missed_out_time_deadline[[
    "part_id", "part_type", "ready_hr", "deadline_hr",
    "cure_time_hr", "priority", "recipe_id"
]])

### Visualize The Two-Autoclave Schedule

Now the timeline shows how the second autoclave creates room for parts that missed the deadline in the first scenario.

In [ ]:
plot_autoclave_schedule(two_ac_solution, "Two-Autoclave Cure Schedule")

### Compare One Autoclave vs Two Autoclaves

This is the business reason for running the extension.

The second autoclave should recover more parts because it adds capacity and allows different compatible cure recipes to run in the same time window.

In [ ]:
comparison = pd.DataFrame([
    {
        "scenario": "One autoclave",
        "cured_parts": len(solution),
        "missed_out_time_deadline": len(missed_out_time_deadline),
        "objective_value": problem.ObjValue,
        "solve_seconds": problem.SolveTime,
    },
    {
        "scenario": "Two autoclaves",
        "cured_parts": len(two_ac_solution),
        "missed_out_time_deadline": len(two_ac_missed_out_time_deadline),
        "objective_value": two_autoclave_problem.ObjValue,
        "solve_seconds": two_autoclave_problem.SolveTime,
    },
])

comparison["objective_value"] = comparison["objective_value"].round(2)
comparison["solve_seconds"] = comparison["solve_seconds"].round(3)

display(comparison)

With this dataset, the one-autoclave model leaves a small number of parts not cured before their out-time deadline.

The two-autoclave model cures all parts within the same timing rules.

That is the key lesson: adding another autoclave changes the feasible schedule, not just the size of the equipment list.

## Notes For A Soft-Deadline Extension Later

A soft-deadline extension is the version where every part must be placed into a schedule.

That changes the formulation:

- `<= 1` becomes `== 1` for each part.
- late completion may be allowed as a penalty instead of blocked.
- we add a lateness or missed-deadline variable for each part.
- the objective becomes something like `minimize total weighted missed-deadline penalty`.

For this workshop, we are **not** adding a separate `due_hr` unless real data gives us one.

If `deadline_hr` truly represents hard out-time expiration, then the soft-deadline model should be treated carefully. A late part may represent scrap, escalation, or recovery cost rather than a valid cured part.

Possible future data additions:

- explicit `recipe_id`
- `material_ready_hr`
- setup/changeover time
- maintenance downtime
- capacity by area or footprint instead of just part count

For now, this notebook teaches the first clean hard-deadline formulation plus the two-autoclave comparison.